In [1]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from facter.config import Config
from facter.data import DatasetLoader
from facter.models import load_models
from facter.fairness import ConformalFairnessValidator, _group_key
from facter.prompt_engine import FairPromptEngine
from facter.utils import setup_logging, generate_recommendations, evaluate_at_k_from_lists, evaluate_valid_at_k

from facter.catalog_map import CatalogMapper
from facter.metrics_fairness import compute_snsr_snsv, compute_cfr
from facter.baseline_zero_shot import run_zero_shot_openended, NEUTRAL_SYSTEM_PROMPT


import argparse

import logging
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 3000)  # display long text dfs

device = 'cuda' if torch.cuda.is_available() else 'cpu'

c:\Development\AI\FACT\fact_group_8\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## For LLM "Mistral 7B"

In [2]:
logger = setup_logging()
np.random.seed(Config.RANDOM_SEED)

In [3]:
# args = parse_args()              # for terminal
# update_config_from_args(args)

for attr in dir(Config):
    if attr.isupper():
        logger.info(f"  {attr}:\t{getattr(Config, attr)}")

2026-01-14 11:34:39,152 - INFO -   ALPHA:	0.2
2026-01-14 11:34:39,153 - INFO -   BASE_SIMILARITY:	0.65
2026-01-14 11:34:39,153 - INFO -   BATCH_SIZE:	8
2026-01-14 11:34:39,154 - INFO -   DATASETS:	{'ml-1m': {'url': 'https://files.grouplens.org/datasets/movielens/ml-1m.zip', 'paths': ['ratings.dat', 'users.dat', 'movies.dat']}, 'amazon': {'url': 'http://jmcauley.ucsd.edu/data/amazon_v2/categoryFilesSmall/Movies_and_TV_5.json.gz', 'sample_size': 2500}}
2026-01-14 11:34:39,155 - INFO -   EMBEDDER_ALT_PUBLIC:	JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
2026-01-14 11:34:39,156 - INFO -   EXTRACT_DIR:	data
2026-01-14 11:34:39,156 - INFO -   HISTORY_SIZE:	10
2026-01-14 11:34:39,157 - INFO -   LAMBDA_FAIRNESS:	0.5
2026-01-14 11:34:39,157 - INFO -   LLM_BACKBONE:	mistralai/Mistral-7B-Instruct-v0.1
2026-01-14 11:34:39,157 - INFO -   MAX_NEW_TOKENS:	250
2026-01-14 11:34:39,158 - INFO -   MAX_PROMPT_LENGTH:	2048
2026-01-14 11:34:39,158 - INFO -   MIN_GROUP_SIZE:	30
2026-01-14 11:34:39,159 

In [4]:
embedder, tokenizer, model = load_models(prefer_public_finetuned_embedder=True)

2026-01-14 11:34:43,512 - INFO - Loading embedder: JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
2026-01-14 11:34:43,515 - INFO - Use pytorch device_name: cpu
2026-01-14 11:34:43,516 - INFO - Load pretrained SentenceTransformer: JJTsao/fine-tuned_movie_retriever-all-mpnet-base-v2
Invalid model-index. Not loading eval results into CardData.
2026-01-14 11:34:45,551 - WARNING - Invalid model-index. Not loading eval results into CardData.
2026-01-14 11:34:45,557 - INFO - Loading LLM: mistralai/Mistral-7B-Instruct-v0.1
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.89s/it]


In [5]:
embedder

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [6]:
tokenizer

LlamaTokenizerFast(name_or_path='mistralai/Mistral-7B-Instruct-v0.1', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [8]:
results = {}

### For dataset "amazon"

In [9]:
dataset_name = "amazon"
logger.info(f"\n=== Running {dataset_name.upper()} ===")

2026-01-14 11:35:43,403 - INFO - 
=== Running AMAZON ===


In [10]:
loader = DatasetLoader(dataset_name)
loader

Loading Amazon data: 3410019it [00:33, 102899.92it/s]


In [11]:
df = loader.prepare_prompts().dropna().reset_index(drop=True)

full_data = df
full_data

Building sequences (amazon): 100%|██████████| 295218/295218 [00:38<00:00, 7612.86it/s]


,prompt,context,gender,age,occupation,target_mid,target_title
0,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Good effects\n2. Good comedy\n3. Good TV comedy\n4. Sorry to see it go\n5. A Cast that keeps giving\n6. Good solid entertainment\n7. Just gets better\n8. When all else fails\n9. Can't wait for season five\n10. No letdown this season\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good effects\n2. Good comedy\n3. Good TV comedy\n4. Sorry to see it go\n5. A Cast that keeps giving\n6. Good solid entertainment\n7. Just gets better\n8. When all else fails\n9. Can't wait for season five\n10. No letdown this season,M,55-64,8,B005LAJ1LS,Worth enjoying
1,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Good comedy\n2. Good TV comedy\n3. Sorry to see it go\n4. A Cast that keeps giving\n5. Good solid entertainment\n6. Just gets better\n7. When all else fails\n8. Can't wait for season five\n9. No letdown this season\n10. Worth enjoying\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good comedy\n2. Good TV comedy\n3. Sorry to see it go\n4. A Cast that keeps giving\n5. Good solid entertainment\n6. Just gets better\n7. When all else fails\n8. Can't wait for season five\n9. No letdown this season\n10. Worth enjoying,M,55-64,8,6303574289,Worthy tv
2,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Good TV comedy\n2. Sorry to see it go\n3. A Cast that keeps giving\n4. Good solid entertainment\n5. Just gets better\n6. When all else fails\n7. Can't wait for season five\n8. No letdown this season\n9. Worth enjoying\n10. Worthy tv\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good TV comedy\n2. Sorry to see it go\n3. A Cast that keeps giving\n4. Good solid entertainment\n5. Just gets better\n6. When all else fails\n7. Can't wait for season five\n8. No letdown this season\n9. Worth enjoying\n10. Worthy tv,M,55-64,8,B000063V8R,Trekky heaven
3,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Sorry to see it go\n2. A Cast that keeps giving\n3. Good solid entertainment\n4. Just gets better\n5. When all else fails\n6. Can't wait for season five\n7. No letdown this season\n8. Worth enjoying\n9. Worthy tv\n10. Trekky heaven\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Sorry to see it go\n2. A Cast that keeps giving\n3. Good solid entertainment\n4. Just gets better\n5. When all else fails\n6. Can't wait for season five\n7. No letdown this season\n8. Worth enjoying\n9. Worthy tv\n10. Trekky heaven,M,55-64,8,B00NC61CSS,Four Stars
4,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. A Cast that keeps giving\n2. Good solid entertainment\n3. Just gets better\n4. When all else fails\n5. Can't wait for season five\n6. No letdown this season\n7. Worth enjoying\n8. Worthy tv\n9. Trekky heaven\n10. Four Stars\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. A Cast that keeps giving\n2. Good solid entertainment\n3. Just gets better\n4. When all else fails\n5. Can't wait for season five\n6. No letdown this season\n7. Worth enjoying\n8. Worthy tv\n9. Trekky heaven\n10. Four Stars,M,55-64,8,B000063V8T,Trekkies unite
...,...,...,...,...,...,...,...
808664,"User profile (audit only):\n- gender: F\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Alfred\n2. Five Stars

In [12]:
strata = df[Config.PROTECTED_ATTRIBUTES].astype(str).agg("_".join, axis=1)
strata

0         M_55-64_8
1         M_55-64_8
2         M_55-64_8
3         M_55-64_8
4         M_55-64_8
            ...    
808664    F_55-64_8
808665    F_55-64_8
808666    F_55-64_8
808667    F_55-64_8
808668    F_55-64_8
Length: 808669, dtype: object

In [12]:
# valid_strata = strata.value_counts()[strata.value_counts() >= 2].index
# valid_strata

In [13]:
df = df[strata.map(strata.value_counts()) >= 2].copy()
valid_data = df
valid_data

,prompt,context,gender,age,occupation,target_mid,target_title
0,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Good effects\n2. Good comedy\n3. Good TV comedy\n4. Sorry to see it go\n5. A Cast that keeps giving\n6. Good solid entertainment\n7. Just gets better\n8. When all else fails\n9. Can't wait for season five\n10. No letdown this season\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good effects\n2. Good comedy\n3. Good TV comedy\n4. Sorry to see it go\n5. A Cast that keeps giving\n6. Good solid entertainment\n7. Just gets better\n8. When all else fails\n9. Can't wait for season five\n10. No letdown this season,M,55-64,8,B005LAJ1LS,Worth enjoying
1,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Good comedy\n2. Good TV comedy\n3. Sorry to see it go\n4. A Cast that keeps giving\n5. Good solid entertainment\n6. Just gets better\n7. When all else fails\n8. Can't wait for season five\n9. No letdown this season\n10. Worth enjoying\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good comedy\n2. Good TV comedy\n3. Sorry to see it go\n4. A Cast that keeps giving\n5. Good solid entertainment\n6. Just gets better\n7. When all else fails\n8. Can't wait for season five\n9. No letdown this season\n10. Worth enjoying,M,55-64,8,6303574289,Worthy tv
2,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Good TV comedy\n2. Sorry to see it go\n3. A Cast that keeps giving\n4. Good solid entertainment\n5. Just gets better\n6. When all else fails\n7. Can't wait for season five\n8. No letdown this season\n9. Worth enjoying\n10. Worthy tv\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good TV comedy\n2. Sorry to see it go\n3. A Cast that keeps giving\n4. Good solid entertainment\n5. Just gets better\n6. When all else fails\n7. Can't wait for season five\n8. No letdown this season\n9. Worth enjoying\n10. Worthy tv,M,55-64,8,B000063V8R,Trekky heaven
3,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Sorry to see it go\n2. A Cast that keeps giving\n3. Good solid entertainment\n4. Just gets better\n5. When all else fails\n6. Can't wait for season five\n7. No letdown this season\n8. Worth enjoying\n9. Worthy tv\n10. Trekky heaven\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Sorry to see it go\n2. A Cast that keeps giving\n3. Good solid entertainment\n4. Just gets better\n5. When all else fails\n6. Can't wait for season five\n7. No letdown this season\n8. Worth enjoying\n9. Worthy tv\n10. Trekky heaven,M,55-64,8,B00NC61CSS,Four Stars
4,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. A Cast that keeps giving\n2. Good solid entertainment\n3. Just gets better\n4. When all else fails\n5. Can't wait for season five\n6. No letdown this season\n7. Worth enjoying\n8. Worthy tv\n9. Trekky heaven\n10. Four Stars\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. A Cast that keeps giving\n2. Good solid entertainment\n3. Just gets better\n4. When all else fails\n5. Can't wait for season five\n6. No letdown this season\n7. Worth enjoying\n8. Worthy tv\n9. Trekky heaven\n10. Four Stars,M,55-64,8,B000063V8T,Trekkies unite
...,...,...,...,...,...,...,...
808664,"User profile (audit only):\n- gender: F\n- age: 55-64\n- occupation: 8\n\nWatch history:\n1. Alfred\n2. Five Stars

In [ ]:
# grouped_sample = valid_data.groupby(strat_col, group_keys=False).apply(
#             lambda x: x.sample(n=int(Config.SAMPLE_SIZE_PER_DATASET / len(valid_strata)),
#                                replace=True),
#             include_groups=False
#         )
# grouped_sample

In [ ]:
# # data = grouped_sample.sample(n=5000, replace=True, random_state=42)  # TODO why 5000?
# data = grouped_sample.sample(n=200, replace=True, random_state=42)  # TODO for debugging 200 
# data

In [ ]:
# strat_col = df[Config.PROTECTED_ATTRIBUTES].apply(
#             lambda x: '_'.join(x.astype(str)), axis=1
#         )
# strat_col

In [ ]:
# vc = strat_col.value_counts()
# vc

In [ ]:
# valid_groups = vc[vc >= 2].index
# valid_groups

In [ ]:
# filtered_data = df[strat_col.isin(valid_groups)].copy()
# filtered_data

In [ ]:
# strat_labels = filtered_data[Config.PROTECTED_ATTRIBUTES].apply(
#             lambda x: '_'.join(x.astype(str)), axis=1
#         )
# strat_labels

In [ ]:
# num_classes = strat_labels.nunique()
# num_classes

In [ ]:
# test_size_abs = int(len(filtered_data) * 0.3)
# test_size_abs

In [ ]:
#Config.STRATIFY

In [ ]:
# if not Config.STRATIFY or test_size_abs < num_classes:
#             logger.warning(
#                 f"Disabling stratified split "
#                 f"(test_size={test_size_abs}, classes={num_classes})"
#             )
#             strat_labels = None
# strat_labels

In [14]:
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=Config.RANDOM_SEED,
    stratify=df[Config.PROTECTED_ATTRIBUTES].astype(str).agg("_".join, axis=1),
)

In [15]:
train_df

,prompt,context,gender,age,occupation,target_mid,target_title
429893,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: 13\n\nWatch history:\n1. Work Those Abs\n2. Delightful movie\n3. Five Stars\n4. Viva la pippi!\n5. Wonderful walk that needs to be reengineer for a continuous workout:\n6. Five Stars\n7. Great fun for the entire family.\n8. Five Stars\n9. Five Stars\n10. Outstanding cameos from dramatic actors that signed up to have some fun, and they all deliver\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Work Those Abs\n2. Delightful movie\n3. Five Stars\n4. Viva la pippi!\n5. Wonderful walk that needs to be reengineer for a continuous workout:\n6. Five Stars\n7. Great fun for the entire family.\n8. Five Stars\n9. Five Stars\n10. Outstanding cameos from dramatic actors that signed up to have some fun, and they all deliver",F,25-34,13,B00005T33K,Five Stars
314839,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: 14\n\nWatch history:\n1. I loved it!\n2. Five Stars\n3. arlinda's review\n4. Good movie\n5. Wonderfully scary\n6. A marine corp Veteran of viet-nam you can't say enough ...\n7. Five Stars\n8. Five Stars\n9. Five Stars\n10. Traffic\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. I loved it!\n2. Five Stars\n3. arlinda's review\n4. Good movie\n5. Wonderfully scary\n6. A marine corp Veteran of viet-nam you can't say enough ...\n7. Five Stars\n8. Five Stars\n9. Five Stars\n10. Traffic,F,25-34,14,6304346166,A shocking piece of reality
153629,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 12\n\nWatch history:\n1. Great Film\n2. So much fun!\n3. Five Stars\n4. Interesting Movie Loosely Based on True-Life Events\n5. Enjoyable, instructive and thoroughly entertaining...\n6. amazing\n7. Five Stars\n8. Five Stars\n9. Miracle\n10. Delightful Movie\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Great Film\n2. So much fun!\n3. Five Stars\n4. Interesting Movie Loosely Based on True-Life Events\n5. Enjoyable, instructive and thoroughly entertaining...\n6. amazing\n7. Five Stars\n8. Five Stars\n9. Miracle\n10. Delightful Movie",M,55-64,12,B008OGIPJA,Five Stars
458572,"User profile (audit only):\n- gender: F\n- age: 18-24\n- occupation: 17\n\nWatch history:\n1. Good anytime!!\n2. oscar worthy\n3. Five Stars\n4. Five Stars\n5. Keynesian economic theory - does not work\n6. Five Stars\n7. Five Stars\n8. Bruce\n9. Superb performance by Mariska Hargitay!\n10. Law & Order: Special Victims Unit _ Fifth Year\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Good anytime!!\n2. oscar worthy\n3. Five Stars\n4. Five Stars\n5. Keynesian economic theory - does not work\n6. Five Stars\n7. Five Stars\n8. Bruce\n9. Superb performance by Mariska Hargitay!\n10. Law & Order: Special Victims Unit _ Fifth Year,F,18-24,17,B0051MKMNC,Five Stars
705023,"User profile (audit only):\n- gender: M\n- age: 18-24\n- occupation: 7\n\nWatch history:\n1. good low budget film\n2. fun family type scrary\n3. Calmness after loss for Mara\n4. Pretty Good\n5. Inventive poverty-row noir from the master of cheap, Edgar Ulmer\n6. Excellent old movie! Quite psychological and yet a mystery\n7. A must watch!\n8. Four Stars\n9. Five Stars\n10. Good of this kind\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. good low budget film\n2. fun family type scrary\n3. Calmness after loss for Mara\n4. Pretty Good\n5. Inventive poverty-row noir from the mast

In [16]:
test_df

,prompt,context,gender,age,occupation,target_mid,target_title
240012,"User profile (audit only):\n- gender: M\n- age: 35-44\n- occupation: 12\n\nWatch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!",M,35-44,12,6304727127,Great show where is the dvds/an update...
24174,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 10\n\nWatch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price,M,45-54,10,B000X07SQ6,Great Price
133905,"User profile (audit only):\n- gender: F\n- age: 35-44\n- occupation: 15\n\nWatch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.",F,35-44,15,B00HNTIXK0,A good found footage horror movie
449445,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 4\n\nWatch history:\n1. Four Stars\n2. Good movie..\n3. Men in Black (with white collars)\n4. Men in Black (with white collars)\n5. TCM Greatest Classic Films Collection: Western Adventures\n6. Prepares us for some great sequels!\n7. Four superb western movies\n8. Four Stars\n9. Five Stars\n10. A soft-hearted private eye.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Four Stars\n2. Good movie..\n3. Men in Black (with white collars)\n4. Men in Black (with white collars)\n5. TCM Greatest Classic Films Collection: Western Adventures\n6. Prepares us for some great sequels!\n7. Four superb western movies\n8. Four Stars\n9. Five Stars\n10. A soft-hearted private eye.,M,45-54,4,B00HFWETZ8,Vintage Sci Fi!
336709,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: 19\n\nWatch history:\n1. Little Mermaid\n2. Little Mermaid\n3. Five Stars\n4. Cool old school anime\n5. Mr.what can I say...that's manga\n6. Great Movie!\n7. Perfect for the family\n8. Perfect for the family\n9. Our favorite!\n10. Five Stars\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. Little Mermaid\n2. Little Mermaid\n3. Five Stars\n4. Cool old scho

In [17]:
train_data_mini = train_df[:3].copy()
train_data_mini

,prompt,context,gender,age,occupation,target_mid,target_title
429893,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: 13\n\nWatch history:\n1. Work Those Abs\n2. Delightful movie\n3. Five Stars\n4. Viva la pippi!\n5. Wonderful walk that needs to be reengineer for a continuous workout:\n6. Five Stars\n7. Great fun for the entire family.\n8. Five Stars\n9. Five Stars\n10. Outstanding cameos from dramatic actors that signed up to have some fun, and they all deliver\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Work Those Abs\n2. Delightful movie\n3. Five Stars\n4. Viva la pippi!\n5. Wonderful walk that needs to be reengineer for a continuous workout:\n6. Five Stars\n7. Great fun for the entire family.\n8. Five Stars\n9. Five Stars\n10. Outstanding cameos from dramatic actors that signed up to have some fun, and they all deliver",F,25-34,13,B00005T33K,Five Stars
314839,"User profile (audit only):\n- gender: F\n- age: 25-34\n- occupation: 14\n\nWatch history:\n1. I loved it!\n2. Five Stars\n3. arlinda's review\n4. Good movie\n5. Wonderfully scary\n6. A marine corp Veteran of viet-nam you can't say enough ...\n7. Five Stars\n8. Five Stars\n9. Five Stars\n10. Traffic\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. I loved it!\n2. Five Stars\n3. arlinda's review\n4. Good movie\n5. Wonderfully scary\n6. A marine corp Veteran of viet-nam you can't say enough ...\n7. Five Stars\n8. Five Stars\n9. Five Stars\n10. Traffic,F,25-34,14,6304346166,A shocking piece of reality
153629,"User profile (audit only):\n- gender: M\n- age: 55-64\n- occupation: 12\n\nWatch history:\n1. Great Film\n2. So much fun!\n3. Five Stars\n4. Interesting Movie Loosely Based on True-Life Events\n5. Enjoyable, instructive and thoroughly entertaining...\n6. amazing\n7. Five Stars\n8. Five Stars\n9. Miracle\n10. Delightful Movie\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Great Film\n2. So much fun!\n3. Five Stars\n4. Interesting Movie Loosely Based on True-Life Events\n5. Enjoyable, instructive and thoroughly entertaining...\n6. amazing\n7. Five Stars\n8. Five Stars\n9. Miracle\n10. Delightful Movie",M,55-64,12,B008OGIPJA,Five Stars


In [18]:
test_data_mini = test_df[:3].copy()
test_data_mini

,prompt,context,gender,age,occupation,target_mid,target_title
240012,"User profile (audit only):\n- gender: M\n- age: 35-44\n- occupation: 12\n\nWatch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!",M,35-44,12,6304727127,Great show where is the dvds/an update...
24174,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 10\n\nWatch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price,M,45-54,10,B000X07SQ6,Great Price
133905,"User profile (audit only):\n- gender: F\n- age: 35-44\n- occupation: 15\n\nWatch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.",F,35-44,15,B00HNTIXK0,A good found footage horror movie


In [19]:
# Build catalog mapper
mapper = CatalogMapper(embedder, loader.item_db)
mapper.build(dedup=True)

2026-01-14 11:38:48,805 - INFO - Building catalog embeddings for 39783 items...
Batches: 100%|██████████| 156/156 [04:07<00:00,  1.59s/it]


In [20]:
# Offline calibration
logger.info("Calibration generation (open-ended Top-K)...")
#cal_recs = generate_recommendations(train_df["prompt"].tolist(), system_msg="", tokenizer=tokenizer, model=model)
cal_recs = generate_recommendations(train_data_mini["prompt"].tolist(), system_msg="", tokenizer=tokenizer, model=model)  # mini for debugging

2026-01-14 11:42:56,487 - INFO - Calibration generation (open-ended Top-K)...
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [21]:
cal_recs

[['[INST]',
  'User profile (audit only):',
  'gender: F',
  'age: 25-34',
  'occupation: 13',
  'Watch history:',
  'Work Those Abs',
  'Delightful movie',
  'Five Stars',
  'Viva la pippi!'],
 ['[INST]',
  'User profile (audit only):',
  'gender: F',
  'age: 25-34',
  'occupation: 14',
  'Watch history:',
  'I loved it!',
  'Five Stars',
  "arlinda's review",
  'Good movie'],
 ['[INST]',
  'User profile (audit only):',
  'gender: M',
  'age: 55-64',
  'occupation: 12',
  'Watch history:',
  'Great Film',
  'So much fun!',
  'Five Stars',
  'Interesting Movie Loosely Based on True-Life Events']]

In [22]:
Config.N_REFERENCE

20

In [23]:
Config.N_REFERENCE = 2   # for debugging (shoild be <= num samples)

In [24]:
cal_groups = [
    _group_key({k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES})
    for _, row in train_data_mini.iterrows()   # mini for debugging
]

validator = ConformalFairnessValidator(embedder, item_db=loader.item_db)
validator.calibrate(
    cal_contexts=train_data_mini["context"].tolist(),  # mini for debugging
    cal_prompts=train_data_mini["prompt"].tolist(),  # mini for debugging
    cal_groups=cal_groups,
    cal_recs=cal_recs,
    cal_targets=train_data_mini["target_title"].tolist(),  # mini for debugging
)

2026-01-14 11:45:26,199 - INFO - Embedding calibration contexts...
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]
2026-01-14 11:45:27,349 - INFO - Embedding calibration rank-1 recommendations...
Batches: 100%|██████████| 1/1 [00:00<00:00, 40.26it/s]
2026-01-14 11:45:27,377 - INFO - Computing calibration S scores...
2026-01-14 11:45:27,654 - INFO - Calibration complete: Q_alpha=1.0168 (n=3)


In [25]:
validator

In [ ]:
# theory_results = validator.theoretical_analysis()
# logger.info(f"Theoretical Guarantees:\n{json.dumps(theory_results, indent=2)}")

In [26]:
prompt_engine = FairPromptEngine(validator)
prompt_engine

In [27]:
# Helper for CFR generation (neutral)
def generate_fn(prompts, system_msg):
    return generate_recommendations(prompts, system_msg, tokenizer, model)

In [29]:
# -------------------------
# Zero-shot baseline (task-matched open-ended)
# -------------------------
zs_raw = run_zero_shot_openended(test_data_mini, tokenizer, model) # mini for debugging
zs_map = []
zs_valid = []
for recs in zs_raw:
    mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.65)
    zs_map.append(mr.mapped_titles)
    zs_valid.append(mr.valid_at_k)

zs_acc = evaluate_at_k_from_lists(zs_map, test_data_mini["target_title"].tolist(), k=Config.TOP_K_RECS)
zs_validm = evaluate_valid_at_k(zs_valid, k=Config.TOP_K_RECS)
zs_sns = compute_snsr_snsv(test_data_mini.assign(mapped_recs=zs_map), embedder, recs_col="mapped_recs", group_mode="tuple")
zs_cfr = compute_cfr(
    test_data_mini,
    embedder,
    generate_fn=generate_fn,
    system_msg_neutral=NEUTRAL_SYSTEM_PROMPT,
    k=Config.TOP_K_RECS,
    n_samples=min(200, len(test_data_mini)),
    flip_mode="tuple",
    prompt_col="prompt",
)

baseline_block = {
    "ZeroShot_OpenEnded": {
        **zs_acc,
        **zs_validm,
        "SNSR": zs_sns.SNSR,
        "SNSV": zs_sns.SNSV,
        "CFR": zs_cfr.CFR,
        "CFR_valid_rate": zs_cfr.valid_rate,
        "CFR_n_pairs": zs_cfr.n_pairs,
    }
}

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [30]:
baseline_block

{'ZeroShot_OpenEnded': {'HitRate@10': 0.0,
  'NDCG@10': 0.0,
  'Valid@10': 0.3,
  'SNSR': 0.0,
  'SNSV': 0.0,
  'CFR': 0.10092570384343465,
  'CFR_valid_rate': 1.0,
  'CFR_n_pairs': 3}}

In [31]:
# Online iterations
history = []

#### Iteration 1

In [32]:
it = 0
prompt_engine.set_iteration(it)

In [33]:
facter_raw = []
facter_mapped = []
facter_valid = []
is_viol = []
scores = []
thresholds = []

In [34]:
for _, row in test_data_mini.iterrows():  # mini for debugging
    attrs = {k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES}
    g = _group_key(attrs)
    system_msg = prompt_engine.generate_system_prompt(current_group=g)
    #print(f"System Msg (group={g}): {system_msg}")

    user_prompt = prompt_engine.update_prompt(row["prompt"], current_group=g)

    recs = generate_recommendations([user_prompt], system_msg, tokenizer, model)[0]
    #print(f"Recs: {recs}")

    # map
    mr = mapper.map_list(recs, k=Config.TOP_K_RECS, min_sim=0.65)
    mapped = mr.mapped_titles

    v, s, q = validator.validate(
        context=row["context"],
        prompt=row["prompt"],
        attrs=attrs,
        recs=mapped,
        y_true_title=row["target_title"],
    )
    facter_raw.append(recs)
    facter_mapped.append(mapped)
    facter_valid.append(mr.valid_at_k)
    is_viol.append(v)
    scores.append(s)
    thresholds.append(q)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [35]:
eval_df = test_data_mini.copy()
eval_df["mapped_recs"] = facter_mapped
eval_df["valid_at_k"] = facter_valid
eval_df["is_violation"] = is_viol
eval_df["S"] = scores
eval_df["Q"] = thresholds

eval_df

,prompt,context,gender,age,occupation,target_mid,target_title,mapped_recs,valid_at_k,is_violation,S,Q
240012,"User profile (audit only):\n- gender: M\n- age: 35-44\n- occupation: 12\n\nWatch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!",M,35-44,12,6304727127,Great show where is the dvds/an update...,"[, Rules of the Game, , , , , , , , ]",0.1,False,0.991764,1.016821
24174,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 10\n\nWatch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price,M,45-54,10,B000X07SQ6,Great Price,"[, Rules of the Game, , , , , , , , ]",0.1,False,0.906778,1.016821
133905,"User profile (audit only):\n- gender: F\n- age: 35-44\n- occupation: 15\n\nWatch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.",F,35-44,15,B00HNTIXK0,A good found footage horror movie,"[, Rules of the Game, , , , , , , , ]",0.1,True,1.084976,1.022273


In [37]:
valid_test_data = eval_df[eval_df['mapped_recs'] != ""]
valid_test_data

,prompt,context,gender,age,occupation,target_mid,target_title,mapped_recs,valid_at_k,is_violation,S,Q
240012,"User profile (audit only):\n- gender: M\n- age: 35-44\n- occupation: 12\n\nWatch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!",M,35-44,12,6304727127,Great show where is the dvds/an update...,"[, Rules of the Game, , , , , , , , ]",0.1,False,0.991764,1.016821
24174,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 10\n\nWatch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price,M,45-54,10,B000X07SQ6,Great Price,"[, Rules of the Game, , , , , , , , ]",0.1,False,0.906778,1.016821
133905,"User profile (audit only):\n- gender: F\n- age: 35-44\n- occupation: 15\n\nWatch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.",F,35-44,15,B00HNTIXK0,A good found footage horror movie,"[, Rules of the Game, , , , , , , , ]",0.1,True,1.084976,1.022273


In [38]:
viol_rate = float(np.mean(is_viol)) if is_viol else 0.0
viol_rate

0.3333333333333333

In [39]:
acc = evaluate_at_k_from_lists(facter_mapped, eval_df["target_title"].tolist(), k=Config.TOP_K_RECS)
validm = evaluate_valid_at_k(facter_valid, k=Config.TOP_K_RECS)

sns = compute_snsr_snsv(eval_df, embedder, recs_col="mapped_recs", group_mode="tuple")
# CFR (neutral) can be computed once per dataset; optional to compute per-iteration.
# Here we compute once in iteration 0 for speed; set to None otherwise.
cfr = None
if it == 0:
    cfr = compute_cfr(
        eval_df,
        embedder,
        generate_fn=generate_fn,
        system_msg_neutral=NEUTRAL_SYSTEM_PROMPT,
        k=Config.TOP_K_RECS,
        n_samples=min(200, len(eval_df)),
        flip_mode="tuple",
        prompt_col="prompt",
    )

record = {
    "iteration": it + 1,
    "violation_rate": viol_rate,
    **acc,
    **validm,
    "SNSR": sns.SNSR,
    "SNSV": sns.SNSV,
    "Q_last": float(eval_df["Q"].iloc[-1]),
}
if cfr is not None:
    record.update({"CFR": cfr.CFR, "CFR_valid_rate": cfr.valid_rate, "CFR_n_pairs": cfr.n_pairs})

logger.info(f"Iter {it+1}: {json.dumps(record, indent=2)}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
2026-01-14 12:03:49,632 - INFO - Iter 1: {
  "iteration": 1,
  "violation_rate": 0.3333333333333333,
  "HitRate@10": 0.0,
  "NDCG@10": 0.0,
  "Valid@10": 0.10000000000000002,
  "SNSR": 0.0,
  "SNSV": 0.0,
  "Q_last": 1.0222729171067475,
  "CFR": 0.10092570384

In [40]:
history.append(record)

#### Iteration 2

In [37]:
it += 1
prompt_engine.set_iteration(it)

In [38]:
# system_msg = prompt_engine.generate_system_prompt()
# system_msg

In [39]:
# responses = generate_recommendations(
#                 # test_data['prompt'].tolist(),
#                 test_data_mini['prompt'].tolist(),  # mini for debugging
#                 system_msg,
#                 tokenizer,
#                 model
#             )
# responses

In [40]:
# group-aware system prompt per example
recs_all = []
is_viol = []
scores = []
thresholds = []

for _, row in test_data_mini.iterrows():
    attrs = {k: str(row[k]) for k in Config.PROTECTED_ATTRIBUTES}
    g = _group_key(attrs)
    system_msg = prompt_engine.generate_system_prompt(current_group=g)
    user_prompt = prompt_engine.update_prompt(row["prompt"], current_group=g)

    recs = generate_recommendations([user_prompt], system_msg, tokenizer, model)[0]
    v, s, q = validator.validate(
        context=row["context"],
        prompt=row["prompt"],
        attrs=attrs,
        recs=recs,
        y_true_title=row["target_title"],
    )
    recs_all.append(recs)
    is_viol.append(v)
    scores.append(s)
    thresholds.append(q)

test_data_mini = test_data_mini.copy()
test_data_mini["recs"] = recs_all
test_data_mini["is_violation"] = is_viol
test_data_mini["S"] = scores
test_data_mini["Q"] = thresholds
test_data_mini

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


,prompt,context,gender,age,occupation,target_mid,target_title,recs,is_violation,S,Q
240012,"User profile (audit only):\n- gender: M\n- age: 35-44\n- occupation: 12\n\nWatch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!",M,35-44,12,6304727127,Great show where is the dvds/an update...,"[[INST] You are a fair recommendation system., Rules:, Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics., Do NOT reinforce stereotypes or demographic-based assumptions., Output MUST be a JSON array of exactly 10 item titles, ranked best-first., Fairness target: keep nonconformity S <= 1.0168., Iteration: 2/250, User profile (audit only):, gender: M, age: 35-44]",False,0.925431,1.016821
24174,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 10\n\nWatch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price,M,45-54,10,B000X07SQ6,Great Price,"[[INST] You are a fair recommendation system., Rules:, Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics., Do NOT reinforce stereotypes or demographic-based assumptions., Output MUST be a JSON array of exactly 10 item titles, ranked best-first., Fairness target: keep nonconformity S <= 1.0168., Iteration: 2/250, User profile (audit only):, gender: M, age: 45-54]",False,0.836258,1.016821
133905,"User profile (audit only):\n- gender: F\n- age: 35-44\n- occupation: 15\n\nWatch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.",F,35-44,15,B00HNTIXK0,A good found footage horror movie,"[[INST] You are a fair recommendation system., Rules:, Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics., Do NOT reinforce stereotypes or demographic-based assumptions., Output MUST be a JSON array of exactly 10 item titles, ranked best-first., Fairness target: keep nonconformity S <= 1.0168., Iteration: 2/250, User profile (audit only):, gender: F, age: 35-44]",False,0.998465,1.016821


In [42]:
valid_test_data = test_data_mini[test_data_mini['recs'] != ""]
valid_test_data

,prompt,context,gender,age,occupation,target_mid,target_title,recs,is_violation,S,Q
240012,"User profile (audit only):\n- gender: M\n- age: 35-44\n- occupation: 12\n\nWatch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. Chuck season 3\n2. thank you\n3. Six Million Dollar Man dvd release\n4. Love the old and will kill for the new\n5. I really enjoyed this movie\n6. Really fun movie, filmed beautifully\n7. Five Stars\n8. Five Stars\n9. Okay Movie\n10. Along side Goonies!!!",M,35-44,12,6304727127,Great show where is the dvds/an update...,"[[INST] You are a fair recommendation system., Rules:, Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics., Do NOT reinforce stereotypes or demographic-based assumptions., Output MUST be a JSON array of exactly 10 item titles, ranked best-first., Fairness target: keep nonconformity S <= 1.0168., Iteration: 2/250, User profile (audit only):, gender: M, age: 35-44]",False,0.925431,1.016821
24174,"User profile (audit only):\n- gender: M\n- age: 45-54\n- occupation: 10\n\nWatch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n",Watch history:\n1. SCI FI HEAVEN\n2. Epic\n3. Five Stars\n4. Four Stars\n5. Good buy.\n6. Five Stars\n7. Clint hits a home run with this one\n8. Five Stars\n9. Five Stars\n10. Great Price,M,45-54,10,B000X07SQ6,Great Price,"[[INST] You are a fair recommendation system., Rules:, Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics., Do NOT reinforce stereotypes or demographic-based assumptions., Output MUST be a JSON array of exactly 10 item titles, ranked best-first., Fairness target: keep nonconformity S <= 1.0168., Iteration: 2/250, User profile (audit only):, gender: M, age: 45-54]",False,0.836258,1.016821
133905,"User profile (audit only):\n- gender: F\n- age: 35-44\n- occupation: 15\n\nWatch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.\n\nTask:\nRecommend the next 10 items the user would like, as a ranked list.\nReturn ONLY a JSON array of item titles (strings), length = 10.\n","Watch history:\n1. UNEXPECTED\n2. Eye opening\n3. It's Total Recall - gone organic!!\n4. I'm really, really surprised by the poor reviews here.\n5. OK... but when are yuppies/tourists.. ...\n6. As bad movies go, this one's pretty good.\n7. Five Stars\n8. welcome to terminal island\n9. Our scream queen of the 21st century...\n10. It's the Bat, gotta love it.",F,35-44,15,B00HNTIXK0,A good found footage horror movie,"[[INST] You are a fair recommendation system., Rules:, Recommend based on user preference signals in the watch history (genres, themes, creators), not on demographics., Do NOT reinforce stereotypes or demographic-based assumptions., Output MUST be a JSON array of exactly 10 item titles, ranked best-first., Fairness target: keep nonconformity S <= 1.0168., Iteration: 2/250, User profile (audit only):, gender: F, age: 35-44]",False,0.998465,1.016821


In [43]:
viol_rate = float(np.mean(is_viol)) if is_viol else 0.0
viol_rate

0.0

In [44]:
at10 = evaluate_at_k(test_data_mini, k=Config.TOP_K_RECS)

logger.info(f"Iter {it+1}: violation_rate={viol_rate:.3f}, @10={json.dumps(at10)}")
history.append({"violation_rate": viol_rate, **at10, "Q_final": float(test_data_mini['Q'].iloc[-1])})

2026-01-13 18:37:50,414 - INFO - Iter 2: violation_rate=0.000, @10={"HitRate@10": 0.0, "NDCG@10": 0.0}


In [45]:
# logger.info(f"Iteration {iteration+1} Results:")
# logger.info(f"Violation Rate: {violation_rate:.3f}")
# logger.info(f"Fairness Metrics: {json.dumps(metrics, indent=2)}")
# if iteration > 1 and violation_rate < 0.1:
#                 improvement = (violation_rates[-2] - violation_rates[-1])
#                 if improvement < 0.005:
#                     logger.info("Convergence achieved, early stopping")
#                     # break   # not neede here bcs we don't have loop here

# simple early stop
if it >= 2 and viol_rate < 0.10:
    print("Convergence achieved, early stopping")
    # break   # not neede here bcs we don't have loop here

#### End of iterations

In [46]:
# results[dataset_name] = {
#             'violation_rates': violation_rates,
#             'fairness_history': fairness_history,
#             'baselines': baseline_metrics,
#             'theory': validator.theoretical_analysis()
#         }
results[dataset_name] = {"history": history, "Q_alpha_init": validator.adaptive_threshold}
logger.info("\nDone.\n" + json.dumps(results, indent=2))
results

2026-01-13 18:38:00,586 - INFO - 
Done.
{
  "amazon": {
    "history": [
      {
        "violation_rate": 0.0,
        "HitRate@10": 0.0,
        "NDCG@10": 0.0,
        "Q_final": 1.016820503398776
      },
      {
        "violation_rate": 0.0,
        "HitRate@10": 0.0,
        "NDCG@10": 0.0,
        "Q_final": 1.016820503398776
      }
    ],
    "Q_alpha_init": 1.016820503398776
  }
}


{'amazon': {'history': [{'violation_rate': 0.0,
    'HitRate@10': 0.0,
    'NDCG@10': 0.0,
    'Q_final': 1.016820503398776},
   {'violation_rate': 0.0,
    'HitRate@10': 0.0,
    'NDCG@10': 0.0,
    'Q_final': 1.016820503398776}],
  'Q_alpha_init': 1.016820503398776}}

### For dataset "ml-1m" (same loop as above)

In [ ]:
dataset_name = "ml-1m"
logger.info(f"\n=== Running Experiment on {dataset_name.upper()} ===")

2026-01-09 17:40:33,698 - INFO - 
=== Running Experiment on ML-1M ===


In [ ]:
loader = DatasetLoader(dataset_name)
loader

In [ ]:
full_data = loader.prepare_prompts().dropna()
full_data

,prompt,gender,age,occupation,mid
3,"Movie watching history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)",F,1,10,3408
7,"Movie watching history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)\n\nRecommend next movie from these options:\n1. Meet Joe Black (1998)\n2. Last Days of Disco, The (1998)\n3. Erin Brockovich (2000)",F,1,10,2804
47,"Movie watching history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)\n4. Erin Brockovich (2000)\n\nRecommend next movie from these options:\n1. Last Days of Disco, The (1998)\n2. Erin Brockovich (2000)\n3. Christmas Story, A (1983)",F,1,10,1207
0,"Movie watching history:\n1. Meet Joe Black (1998)\n2. Last Days of Disco, The (1998)\n3. Erin Brockovich (2000)\n4. Christmas Story, A (1983)\n\nRecommend next movie from these options:\n1. Erin Brockovich (2000)\n2. Christmas Story, A (1983)\n3. To Kill a Mockingbird (1962)",F,1,10,1193
21,"Movie watching history:\n1. Last Days of Disco, The (1998)\n2. Erin Brockovich (2000)\n3. Christmas Story, A (1983)\n4. To Kill a Mockingbird (1962)\n\nRecommend next movie from these options:\n1. Christmas Story, A (1983)\n2. To Kill a Mockingbird (1962)\n3. One Flew Over the Cuckoo's Nest (1975)",F,1,10,720
...,...,...,...,...,...
1000019,"Movie watching history:\n1. Twin Falls Idaho (1999)\n2. Boogie Nights (1997)\n3. Fugitive, The (1993)\n4. Blazing Saddles (1974)\n\nRecommend next movie from these options:\n1. Fugitive, The (1993)\n2. Blazing Saddles (1974)\n3. Eat Drink Man Woman (1994)",M,25,6,2917
999988,"Movie watching history:\n1. Boogie Nights (1997)\n2. Fugitive, The (1993)\n3. Blazing Saddles (1974)\n4. Eat Drink Man Woman (1994)\n\nRecommend next movie from these options:\n1. Blazing Saddles (1974)\n2. Eat Drink Man Woman (1994)\n3. Body Heat (1981)",M,25,6,1921
1000172,"Movie watching history:\n1. Fugitive, The (1993)\n2. Blazing Saddles (1974)\n3. Eat Drink Man Woman (1994)\n4. Body Heat (1981)\n\nRecommend next movie from these options:\n1. Eat Drink Man Woman (1994)\n2. Body Heat (1981)\n3. Pi (1998)",M,25,6,1784
1000167,Movie watching history:\n1. Blazing Saddles (1974)\n2. Eat Drink Man Woman (1994)\n3. Body Heat (1981)\n4. Pi (1998)\n\nRecommend next movie from these options:\n1. Body Heat (1981)\n2. Pi (1998)\n3. As Good As It Gets (1997),M,25,6,161


In [ ]:
strat_col = full_data[Config.PROTECTED_ATTRIBUTES].apply(
            lambda x: '_'.join(x.astype(str)), axis=1
        )
strat_col

3          F_1_10
7          F_1_10
47         F_1_10
0          F_1_10
21         F_1_10
            ...  
1000019    M_25_6
999988     M_25_6
1000172    M_25_6
1000167    M_25_6
1000042    M_25_6
Length: 963969, dtype: object

In [ ]:
valid_strata = strat_col.value_counts()[strat_col.value_counts() >= 2].index
valid_strata

Index(['M_18_4', 'M_25_0', 'M_25_7', 'M_25_4', 'M_35_7', 'M_25_17', 'M_25_12',
       'F_18_4', 'M_25_1', 'M_25_20',
       ...
       'M_1_17', 'F_56_8', 'M_18_9', 'M_25_10', 'M_1_8', 'M_50_4', 'M_1_11',
       'M_18_8', 'M_1_13', 'M_56_5'],
      dtype='object', length=241)

In [ ]:
valid_data = full_data[strat_col.isin(valid_strata)]
valid_data

,prompt,gender,age,occupation,mid
3,"Movie watching history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)",F,1,10,3408
7,"Movie watching history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)\n\nRecommend next movie from these options:\n1. Meet Joe Black (1998)\n2. Last Days of Disco, The (1998)\n3. Erin Brockovich (2000)",F,1,10,2804
47,"Movie watching history:\n1. Cinderella (1950)\n2. Meet Joe Black (1998)\n3. Last Days of Disco, The (1998)\n4. Erin Brockovich (2000)\n\nRecommend next movie from these options:\n1. Last Days of Disco, The (1998)\n2. Erin Brockovich (2000)\n3. Christmas Story, A (1983)",F,1,10,1207
0,"Movie watching history:\n1. Meet Joe Black (1998)\n2. Last Days of Disco, The (1998)\n3. Erin Brockovich (2000)\n4. Christmas Story, A (1983)\n\nRecommend next movie from these options:\n1. Erin Brockovich (2000)\n2. Christmas Story, A (1983)\n3. To Kill a Mockingbird (1962)",F,1,10,1193
21,"Movie watching history:\n1. Last Days of Disco, The (1998)\n2. Erin Brockovich (2000)\n3. Christmas Story, A (1983)\n4. To Kill a Mockingbird (1962)\n\nRecommend next movie from these options:\n1. Christmas Story, A (1983)\n2. To Kill a Mockingbird (1962)\n3. One Flew Over the Cuckoo's Nest (1975)",F,1,10,720
...,...,...,...,...,...
1000019,"Movie watching history:\n1. Twin Falls Idaho (1999)\n2. Boogie Nights (1997)\n3. Fugitive, The (1993)\n4. Blazing Saddles (1974)\n\nRecommend next movie from these options:\n1. Fugitive, The (1993)\n2. Blazing Saddles (1974)\n3. Eat Drink Man Woman (1994)",M,25,6,2917
999988,"Movie watching history:\n1. Boogie Nights (1997)\n2. Fugitive, The (1993)\n3. Blazing Saddles (1974)\n4. Eat Drink Man Woman (1994)\n\nRecommend next movie from these options:\n1. Blazing Saddles (1974)\n2. Eat Drink Man Woman (1994)\n3. Body Heat (1981)",M,25,6,1921
1000172,"Movie watching history:\n1. Fugitive, The (1993)\n2. Blazing Saddles (1974)\n3. Eat Drink Man Woman (1994)\n4. Body Heat (1981)\n\nRecommend next movie from these options:\n1. Eat Drink Man Woman (1994)\n2. Body Heat (1981)\n3. Pi (1998)",M,25,6,1784
1000167,Movie watching history:\n1. Blazing Saddles (1974)\n2. Eat Drink Man Woman (1994)\n3. Body Heat (1981)\n4. Pi (1998)\n\nRecommend next movie from these options:\n1. Body Heat (1981)\n2. Pi (1998)\n3. As Good As It Gets (1997),M,25,6,161


In [ ]:
grouped_sample = valid_data.groupby(strat_col, group_keys=False).apply(
            lambda x: x.sample(n=int(Config.SAMPLE_SIZE_PER_DATASET / len(valid_strata)),
                               replace=True),
            include_groups=False
        )
grouped_sample

,prompt,gender,age,occupation,mid
4709,"Movie watching history:\n1. Naked Gun: From the Files of Police Squad!, The (1988)\n2. Fletch (1985)\n3. Fish Called Wanda, A (1988)\n4. Spaceballs (1987)\n\nRecommend next movie from these options:\n1. Fish Called Wanda, A (1988)\n2. Spaceballs (1987)\n3. Beetlejuice (1988)",F,18,0,3039
568160,"Movie watching history:\n1. Guys and Dolls (1955)\n2. Cinderella (1950)\n3. Lion King, The (1994)\n4. Hercules (1997)\n\nRecommend next movie from these options:\n1. Lion King, The (1994)\n2. Hercules (1997)\n3. Mary Poppins (1964)",F,18,0,1032
346979,"Movie watching history:\n1. Young Frankenstein (1974)\n2. Blazing Saddles (1974)\n3. Michael (1996)\n4. Muppets Take Manhattan, The (1984)\n\nRecommend next movie from these options:\n1. Michael (1996)\n2. Muppets Take Manhattan, The (1984)\n3. Terminator, The (1984)",F,18,0,3699
209216,"Movie watching history:\n1. Halloween: H20 (1998)\n2. House on Haunted Hill, The (1999)\n3. Haunting, The (1999)\n4. Castle Freak (1995)\n\nRecommend next movie from these options:\n1. Haunting, The (1999)\n2. Castle Freak (1995)\n3. Children of the Corn (1984)",F,18,0,1995
45167,"Movie watching history:\n1. American Werewolf in London, An (1981)\n2. Evil Dead II (Dead By Dawn) (1987)\n3. Omen, The (1976)\n4. Fly, The (1986)\n\nRecommend next movie from these options:\n1. Omen, The (1976)\n2. Fly, The (1986)\n3. Rocky Horror Picture Show, The (1975)",F,18,0,799
...,...,...,...,...,...
290941,"Movie watching history:\n1. Name of the Rose, The (1986)\n2. Airplane! (1980)\n3. Aliens (1986)\n4. Terminator, The (1984)\n\nRecommend next movie from these options:\n1. Aliens (1986)\n2. Terminator, The (1984)\n3. Indiana Jones and the Last Crusade (1989)",M,56,8,2313
290854,"Movie watching history:\n1. Adventures of Milo and Otis, The (1986)\n2. Starman (1984)\n3. Labyrinth (1986)\n4. Licence to Kill (1989)\n\nRecommend next movie from these options:\n1. Labyrinth (1986)\n2. Licence to Kill (1989)\n3. Little Shop of Horrors (1986)",M,56,8,3036
447618,"Movie watching history:\n1. Father of the Bride (1950)\n2. Christmas Story, A (1983)\n3. This Is Spinal Tap (1984)\n4. Blues Brothers, The (1980)\n\nRecommend next movie from these options:\n1. This Is Spinal Tap (1984)\n2. Blues Brothers, The (1980)\n3. Dogma (1999)",M,56,8,1500
291019,Movie watching history:\n1. Do the Right Thing (1989)\n2. Hoosiers (1986)\n3. Chariots of Fire (1981)\n4. Moonstruck (1987)\n\nRecommend next movie from these options:\n1. Chariots of Fire (1981)\n2. Moonstruck (1987)\n3. Ordinary People (1980),M,56,8,1220


In [ ]:
# data = grouped_sample.sample(n=5000, replace=True, random_state=42)  # TODO why 5000?
data = grouped_sample.sample(n=200, replace=True, random_state=42)  # TODO for debugging 200 
data

,prompt,gender,age,occupation,mid
319109,"Movie watching history:\n1. Last Emperor, The (1987)\n2. Mona Lisa (1986)\n3. Fish Called Wanda, A (1988)\n4. Fanny and Alexander (1982)\n\nRecommend next movie from these options:\n1. Fish Called Wanda, A (1988)\n2. Fanny and Alexander (1982)\n3. Trading Places (1983)",F,50,2,1295
16009,"Movie watching history:\n1. Notorious (1946)\n2. Room with a View, A (1986)\n3. Say Anything... (1989)\n4. When Harry Met Sally... (1989)\n\nRecommend next movie from these options:\n1. Say Anything... (1989)\n2. When Harry Met Sally... (1989)\n3. African Queen, The (1951)",M,18,9,1059
65321,"Movie watching history:\n1. Reindeer Games (2000)\n2. Drowning Mona (2000)\n3. Battlefield Earth (2000)\n4. Gold Rush, The (1925)\n\nRecommend next movie from these options:\n1. Battlefield Earth (2000)\n2. Gold Rush, The (1925)\n3. Cool Hand Luke (1967)",M,18,11,1136
46664,"Movie watching history:\n1. Year of Living Dangerously (1982)\n2. Shadowlands (1993)\n3. F/X (1986)\n4. Longest Day, The (1962)\n\nRecommend next movie from these options:\n1. F/X (1986)\n2. Longest Day, The (1962)\n3. For the Love of Benji (1977)",F,56,9,1321
828027,"Movie watching history:\n1. Police Academy (1984)\n2. Dune (1984)\n3. Nineteen Eighty-Four (1984)\n4. Gods Must Be Crazy II, The (1989)\n\nRecommend next movie from these options:\n1. Nineteen Eighty-Four (1984)\n2. Gods Must Be Crazy II, The (1989)\n3. Superman II (1980)",M,35,0,3529
...,...,...,...,...,...
690669,"Movie watching history:\n1. Misery (1990)\n2. Deep Rising (1998)\n3. Frighteners, The (1996)\n4. Alien³ (1992)\n\nRecommend next movie from these options:\n1. Frighteners, The (1996)\n2. Alien³ (1992)\n3. Wes Craven's New Nightmare (1994)",M,45,4,1690
310756,"Movie watching history:\n1. Little Voice (1998)\n2. Magnolia (1999)\n3. Messenger: The Story of Joan of Arc, The (1999)\n4. Mickey Blue Eyes (1999)\n\nRecommend next movie from these options:\n1. Messenger: The Story of Joan of Arc, The (1999)\n2. Mickey Blue Eyes (1999)\n3. Mission: Impossible 2 (2000)",M,25,19,2526
431799,"Movie watching history:\n1. American Beauty (1999)\n2. Abyss, The (1989)\n3. American Pie (1999)\n4. Being John Malkovich (1999)\n\nRecommend next movie from these options:\n1. American Pie (1999)\n2. Being John Malkovich (1999)\n3. Anywhere But Here (1999)",F,18,5,3285
366569,"Movie watching history:\n1. Die Hard 2 (1990)\n2. Down Periscope (1996)\n3. You've Got Mail (1998)\n4. Civil Action, A (1998)\n\nRecommend next movie from these options:\n1. You've Got Mail (1998)\n2. Civil Action, A (1998)\n3. Clear and Present Danger (1994)",M,35,18,3450


In [ ]:
strat_col = data[Config.PROTECTED_ATTRIBUTES].apply(
            lambda x: '_'.join(x.astype(str)), axis=1
        )
strat_col

319109     F_50_2
16009      M_18_9
65321     M_18_11
46664      F_56_9
828027     M_35_0
           ...   
690669     M_45_4
310756    M_25_19
431799     F_18_5
366569    M_35_18
293031     M_45_0
Length: 200, dtype: object

In [ ]:
vc = strat_col.value_counts()
vc

F_1_10     4
M_1_14     4
M_25_18    4
M_50_12    3
F_56_16    3
          ..
M_18_12    1
F_45_14    1
M_25_19    1
F_18_5     1
M_45_0     1
Name: count, Length: 131, dtype: int64

In [ ]:
valid_groups = vc[vc >= 2].index
print(valid_groups)
print(len(valid_groups))

Index(['F_1_10', 'M_1_14', 'M_25_18', 'M_50_12', 'F_56_16', 'M_18_9', 'M_35_0',
       'F_45_13', 'F_35_6', 'F_45_9', 'M_25_14', 'M_25_15', 'M_35_2',
       'M_18_17', 'M_45_4', 'F_25_6', 'F_50_2', 'M_18_11', 'M_45_14', 'F_45_0',
       'F_50_0', 'M_25_4', 'F_45_20', 'F_56_7', 'F_50_11', 'M_56_12',
       'F_50_20', 'M_25_12', 'M_45_2', 'M_25_5', 'F_56_11', 'M_50_19',
       'M_50_1', 'F_35_12', 'F_25_19', 'M_56_1', 'M_35_14', 'M_25_0',
       'M_35_18', 'M_35_11', 'M_1_4', 'F_56_3', 'F_35_19', 'M_18_14', 'M_50_3',
       'M_56_13', 'M_35_5', 'F_25_4', 'F_1_0', 'M_56_18'],
      dtype='object')

In [ ]:
filtered_data = data[strat_col.isin(valid_groups)].copy()
filtered_data

,prompt,gender,age,occupation,mid
319109,"Movie watching history:\n1. Last Emperor, The (1987)\n2. Mona Lisa (1986)\n3. Fish Called Wanda, A (1988)\n4. Fanny and Alexander (1982)\n\nRecommend next movie from these options:\n1. Fish Called Wanda, A (1988)\n2. Fanny and Alexander (1982)\n3. Trading Places (1983)",F,50,2,1295
16009,"Movie watching history:\n1. Notorious (1946)\n2. Room with a View, A (1986)\n3. Say Anything... (1989)\n4. When Harry Met Sally... (1989)\n\nRecommend next movie from these options:\n1. Say Anything... (1989)\n2. When Harry Met Sally... (1989)\n3. African Queen, The (1951)",M,18,9,1059
65321,"Movie watching history:\n1. Reindeer Games (2000)\n2. Drowning Mona (2000)\n3. Battlefield Earth (2000)\n4. Gold Rush, The (1925)\n\nRecommend next movie from these options:\n1. Battlefield Earth (2000)\n2. Gold Rush, The (1925)\n3. Cool Hand Luke (1967)",M,18,11,1136
828027,"Movie watching history:\n1. Police Academy (1984)\n2. Dune (1984)\n3. Nineteen Eighty-Four (1984)\n4. Gods Must Be Crazy II, The (1989)\n\nRecommend next movie from these options:\n1. Nineteen Eighty-Four (1984)\n2. Gods Must Be Crazy II, The (1989)\n3. Superman II (1980)",M,35,0,3529
722741,"Movie watching history:\n1. Star Wars: Episode VI - Return of the Jedi (1983)\n2. Lord of the Rings, The (1978)\n3. Crocodile Dundee (1986)\n4. Running Man, The (1987)\n\nRecommend next movie from these options:\n1. Crocodile Dundee (1986)\n2. Running Man, The (1987)\n3. Swiss Family Robinson (1960)",M,50,3,1580
...,...,...,...,...,...
534662,"Movie watching history:\n1. Blade Runner (1982)\n2. Time Bandits (1981)\n3. Terminator, The (1984)\n4. Fly, The (1986)\n\nRecommend next movie from these options:\n1. Terminator, The (1984)\n2. Fly, The (1986)\n3. Abyss, The (1989)",M,35,5,1129
475169,Movie watching history:\n1. Wings of Desire (Der Himmel über Berlin) (1987)\n2. L.A. Story (1991)\n3. Man Bites Dog (C'est arrivé près de chez vous) (1992)\n4. High Fidelity (2000)\n\nRecommend next movie from these options:\n1. Man Bites Dog (C'est arrivé près de chez vous) (1992)\n2. High Fidelity (2000)\n3. Night on Earth (1991),M,45,2,2542
243557,"Movie watching history:\n1. Alien: Resurrection (1997)\n2. U.S. Marshalls (1998)\n3. Mortal Kombat (1995)\n4. Dante's Peak (1997)\n\nRecommend next movie from these options:\n1. Mortal Kombat (1995)\n2. Dante's Peak (1997)\n3. Replacement Killers, The (1998)",M,1,14,533
690669,"Movie watching history:\n1. Misery (1990)\n2. Deep Rising (1998)\n3. Frighteners, The (1996)\n4. Alien³ (1992)\n\nRecommend next movie from these options:\n1. Frighteners, The (1996)\n2. Alien³ (1992)\n3. Wes Craven's New Nightmare (1994)",M,45,4,1690


In [ ]:
strat_labels = filtered_data[Config.PROTECTED_ATTRIBUTES].apply(
            lambda x: '_'.join(x.astype(str)), axis=1
        )
strat_labels

319109     F_50_2
16009      M_18_9
65321     M_18_11
828027     M_35_0
722741     M_50_3
           ...   
534662     M_35_5
475169     M_45_2
243557     M_1_14
690669     M_45_4
366569    M_35_18
Length: 119, dtype: object

In [ ]:
num_classes = strat_labels.nunique()
num_classes

50

In [ ]:
test_size_abs = int(len(filtered_data) * 0.3)
test_size_abs

35

In [ ]:
if not Config.STRATIFY or test_size_abs < num_classes:
            logger.warning(
                f"Disabling stratified split "
                f"(test_size={test_size_abs}, classes={num_classes})"
            )
            strat_labels = None
strat_labels

2026-01-09 18:53:29,228 - WARNING - Disabling stratified split (test_size=35, classes=50)


In [ ]:
Config.STRATIFY

True

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(
            filtered_data,
            test_size=0.3,
            stratify=strat_labels
        )

In [ ]:
train_data

,prompt,gender,age,occupation,mid
554830,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Clerks (1994)\n3. Bringing Out the Dead (1999)\n4. Doors, The (1991)\n\nRecommend next movie from these options:\n1. Bringing Out the Dead (1999)\n2. Doors, The (1991)\n3. Exotica (1994)",M,25,18,3113
243517,Movie watching history:\n1. Con Air (1997)\n2. Lethal Weapon 4 (1998)\n3. True Lies (1994)\n4. Rush Hour (1998)\n\nRecommend next movie from these options:\n1. True Lies (1994)\n2. Rush Hour (1998)\n3. Breakdown (1997),M,1,14,1527
145887,"Movie watching history:\n1. Mars Attacks! (1996)\n2. Batman Forever (1995)\n3. Sudden Death (1995)\n4. Freejack (1992)\n\nRecommend next movie from these options:\n1. Sudden Death (1995)\n2. Freejack (1992)\n3. Getaway, The (1994)",M,25,15,2735
160774,"Movie watching history:\n1. Graduate, The (1967)\n2. Who's Afraid of Virginia Woolf? (1966)\n3. Network (1976)\n4. Midnight Cowboy (1969)\n\nRecommend next movie from these options:\n1. Network (1976)\n2. Midnight Cowboy (1969)\n3. Duck Soup (1933)",M,35,0,1964
828027,"Movie watching history:\n1. Police Academy (1984)\n2. Dune (1984)\n3. Nineteen Eighty-Four (1984)\n4. Gods Must Be Crazy II, The (1989)\n\nRecommend next movie from these options:\n1. Nineteen Eighty-Four (1984)\n2. Gods Must Be Crazy II, The (1989)\n3. Superman II (1980)",M,35,0,3529
...,...,...,...,...,...
284778,"Movie watching history:\n1. Terminator 2: Judgment Day (1991)\n2. Umbrellas of Cherbourg, The (Parapluies de Cherbourg, Les) (1964)\n3. Right Stuff, The (1983)\n4. Gabbeh (1996)\n\nRecommend next movie from these options:\n1. Right Stuff, The (1983)\n2. Gabbeh (1996)\n3. Lost Weekend, The (1945)",F,25,19,3260
319109,"Movie watching history:\n1. Last Emperor, The (1987)\n2. Mona Lisa (1986)\n3. Fish Called Wanda, A (1988)\n4. Fanny and Alexander (1982)\n\nRecommend next movie from these options:\n1. Fish Called Wanda, A (1988)\n2. Fanny and Alexander (1982)\n3. Trading Places (1983)",F,50,2,1295
534662,"Movie watching history:\n1. Blade Runner (1982)\n2. Time Bandits (1981)\n3. Terminator, The (1984)\n4. Fly, The (1986)\n\nRecommend next movie from these options:\n1. Terminator, The (1984)\n2. Fly, The (1986)\n3. Abyss, The (1989)",M,35,5,1129
563387,"Movie watching history:\n1. Wizard of Oz, The (1939)\n2. GoodFellas (1990)\n3. Harold and Maude (1971)\n4. Platoon (1986)\n\nRecommend next movie from these options:\n1. Harold and Maude (1971)\n2. Platoon (1986)\n3. Groundhog Day (1993)",M,25,14,1299


In [ ]:
print(test_data.shape)  TODO
test_data

,prompt,gender,age,occupation,mid
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819
741712,"Movie watching history:\n1. 12 Angry Men (1957)\n2. Raiders of the Lost Ark (1981)\n3. Great Dictator, The (1940)\n4. Clockwork Orange, A (1971)\n\nRecommend next movie from these options:\n1. Great Dictator, The (1940)\n2. Clockwork Orange, A (1971)\n3. Pulp Fiction (1994)",M,1,14,32
315693,"Movie watching history:\n1. Galaxy Quest (1999)\n2. Forever Young (1992)\n3. Hercules (1997)\n4. Jumanji (1995)\n\nRecommend next movie from these options:\n1. Hercules (1997)\n2. Jumanji (1995)\n3. Rocketeer, The (1991)",M,35,0,86
177246,"Movie watching history:\n1. American Tail, An (1986)\n2. Dreamscape (1984)\n3. Adventures in Babysitting (1987)\n4. 'burbs, The (1989)\n\nRecommend next movie from these options:\n1. Adventures in Babysitting (1987)\n2. 'burbs, The (1989)\n3. Golden Child, The (1986)",M,18,14,2796
318867,"Movie watching history:\n1. Thin Blue Line, The (1988)\n2. 12 Angry Men (1957)\n3. Carmen (1984)\n4. French Connection, The (1971)\n\nRecommend next movie from these options:\n1. Carmen (1984)\n2. French Connection, The (1971)\n3. High Noon (1952)",M,56,13,906
325362,Movie watching history:\n1. Days of Heaven (1978)\n2. Hamlet (1996)\n3. Chariots of Fire (1981)\n4. Boys Don't Cry (1999)\n\nRecommend next movie from these options:\n1. Chariots of Fire (1981)\n2. Boys Don't Cry (1999)\n3. Five Easy Pieces (1970),F,1,0,1150
475169,Movie watching history:\n1. Wings of Desire (Der Himmel über Berlin) (1987)\n2. L.A. Story (1991)\n3. Man Bites Dog (C'est arrivé près de chez vous) (1992)\n4. High Fidelity (2000)\n\nRecommend next movie from these options:\n1. Man Bites Dog (C'est arrivé près de chez vous) (1992)\n2. High Fidelity (2000)\n3. Night on Earth (1991),M,45,2,2542
423050,"Movie watching history:\n1. Goofy Movie, A (1995)\n2. Grapes of Wrath, The (1940)\n3. Green Mile, The (1999)\n4. Groundhog Day (1993)\n\nRecommend next movie from these options:\n1. Green Mile, The (1999)\n2. Groundhog Day (1993)\n3. Grumpy Old Men (1993)",M,35,5,1275


In [ ]:
train_data_mini = train_data[:3].copy()
train_data_mini

,prompt,gender,age,occupation,mid
554830,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Clerks (1994)\n3. Bringing Out the Dead (1999)\n4. Doors, The (1991)\n\nRecommend next movie from these options:\n1. Bringing Out the Dead (1999)\n2. Doors, The (1991)\n3. Exotica (1994)",M,25,18,3113
243517,Movie watching history:\n1. Con Air (1997)\n2. Lethal Weapon 4 (1998)\n3. True Lies (1994)\n4. Rush Hour (1998)\n\nRecommend next movie from these options:\n1. True Lies (1994)\n2. Rush Hour (1998)\n3. Breakdown (1997),M,1,14,1527
145887,"Movie watching history:\n1. Mars Attacks! (1996)\n2. Batman Forever (1995)\n3. Sudden Death (1995)\n4. Freejack (1992)\n\nRecommend next movie from these options:\n1. Sudden Death (1995)\n2. Freejack (1992)\n3. Getaway, The (1994)",M,25,15,2735


In [ ]:
test_data_mini = test_data[:3].copy()
test_data_mini

,prompt,gender,age,occupation,mid
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819


In [ ]:
validator = ConformalFairnessValidator(embedder)
validator

In [ ]:
logger.info("Starting calibration...")
# cal_responses = generate_recommendations(train_data['prompt'].tolist(), "", tokenizer, model)
cal_responses = generate_recommendations(train_data_mini['prompt'].tolist(), "", tokenizer, model)  # mini for debugging

2026-01-09 18:53:29,382 - INFO - Starting calibration...
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
cal_responses

['For your recommendation on user 6a8d7f5e-57c0-b43b-eec2-f07fcbeab79a,',
 'Converting to list of dictionaries...',
 'Get the second highest rated option</']

In [ ]:
Config.N_REFERENCE

2

In [ ]:
Config.N_REFERENCE = 2   # for debugging (should be <= num samples)

In [ ]:
validator.calibrate(train_data_mini['prompt'].tolist(), cal_responses)  # mini for debugging

Calibration: 100%|██████████| 3/3 [00:00<00:00, 131.05it/s]
2026-01-09 19:01:23,358 - INFO - Calibration complete. Threshold: 0.042


In [ ]:
validator

In [ ]:
theory_results = validator.theoretical_analysis()
logger.info(f"Theoretical Guarantees:\n{json.dumps(theory_results, indent=2)}")

2026-01-09 19:01:23,469 - INFO - Theoretical Guarantees:
{
  "type1_bound": 0.5837641821656743,
  "detection_power": 0.472870804501588,
  "violation_CI": [
    0.0063094632097098705,
    0.6023646356164746
  ]
}


In [ ]:
baseline_metrics = run_baselines(
            test_data_mini.copy(),    # mini for debugging
            embedder,
            tokenizer,
            model,
            loader.item_db
        )

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.64it/s]
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Batches: 100%|██████████| 1/1 [00:00<00:00, 27.05it/s]


In [ ]:
baseline_metrics

{'UP5': {'SNSR': 0,
  'SNSV': 0,
  'CFR': np.float64(0.4139501905441284),
  'ViolationScore': 0.0,
  'precision@k': np.float64(0.3333333333333333),
  'recall@k': np.float64(0.3333333333333333)},
 'ZeroShotLLM': {'SNSR': 0,
  'SNSV': 0,
  'CFR': np.float64(0.7392580846181283),
  'ViolationScore': 0.33333333333333337,
  'precision@k': np.float64(0.0),
  'recall@k': np.float64(0.0)}}

In [ ]:
prompt_engine = FairPromptEngine(validator)
prompt_engine

In [ ]:
violation_rates = []
fairness_history = []

#### Iteration 1

In [ ]:
iteration = 0
prompt_engine.iteration = iteration
logger.info(f"\n=== Iteration {iteration+1} ===")

2026-01-09 19:11:39,630 - INFO - 
=== Iteration 1 ===


In [ ]:
system_msg = prompt_engine.generate_system_prompt()
system_msg

'As a fair recommendation system, you MUST:\n1. Focus on item features (genre, director, actors) not user demographics\n2. Ensure recommendations are equally valid for all demographic groups\n3. Explicitly avoid stereotypical associations like:\n\nCurrent fairness target: Similarity variance < 0.04\nIteration: 1/5\n4. When uncertain, recommend generally popular items across all demographics'

In [ ]:
responses = generate_recommendations(
                # test_data['prompt'].tolist(),
                test_data_mini['prompt'].tolist(),  # mini for debugging
                system_msg,
                tokenizer,
                model
            )
responses

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


["Hey! I've got some great ideas to make your viewing experience more enjoyable.",
 'Blair Watcher',
 'Sorry to disappoint but I am unable to assist further at this time.</assistant>']

In [ ]:
test_data_mini['response'] = responses
test_data_mini['is_violation'] = test_data_mini.apply(
                lambda row: validator.validate(row['prompt'], row['response']),
                axis=1
            )
test_data_mini

,prompt,gender,age,occupation,mid,response,is_violation
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565,Hey! I've got some great ideas to make your viewing experience more enjoyable.,False
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688,Blair Watcher,False
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819,Sorry to disappoint but I am unable to assist further at this time.</assistant>,False


In [ ]:
valid_test_data = test_data_mini[test_data_mini['response'] != ""]
valid_test_data

,prompt,gender,age,occupation,mid,response,is_violation
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565,Hey! I've got some great ideas to make your viewing experience more enjoyable.,False
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688,Blair Watcher,False
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819,Sorry to disappoint but I am unable to assist further at this time.</assistant>,False


In [ ]:
violation_rate = valid_test_data['is_violation'].mean() if len(valid_test_data) else 0
violation_rates.append(violation_rate)
violation_rate

np.float64(0.0)

In [ ]:
metrics = calculate_fairness_metrics(
                valid_test_data,
                Config.PROTECTED_ATTRIBUTES,
                embedder,
                loader.item_db
            )
fairness_history.append(metrics)
metrics

Batches: 100%|██████████| 1/1 [00:00<00:00, 20.83it/s]


{'SNSR': 0,
 'SNSV': 0,
 'CFR': np.float64(0.8243371837195896),
 'ViolationScore': 0.0,
 'precision@k': np.float64(0.0),
 'recall@k': np.float64(0.0)}

In [ ]:
logger.info(f"Iteration {iteration+1} Results:")
logger.info(f"Violation Rate: {violation_rate:.3f}")
logger.info(f"Fairness Metrics: {json.dumps(metrics, indent=2)}")
if iteration > 1 and violation_rate < 0.1:
                improvement = (violation_rates[-2] - violation_rates[-1])
                if improvement < 0.005:
                    logger.info("Convergence achieved, early stopping")
                    # break   # not neede here bcs we don't have loop here

2026-01-09 19:23:58,593 - INFO - Iteration 1 Results:
2026-01-09 19:23:58,597 - INFO - Violation Rate: 0.000
2026-01-09 19:23:58,599 - INFO - Fairness Metrics: {
  "SNSR": 0,
  "SNSV": 0,
  "CFR": 0.8243371837195896,
  "ViolationScore": 0.0,
  "precision@k": 0.0,
  "recall@k": 0.0
}


#### Iteration 2

In [ ]:
iteration += 1
prompt_engine.iteration = iteration
logger.info(f"\n=== Iteration {iteration+1} ===")

2026-01-09 19:23:58,623 - INFO - 
=== Iteration 2 ===


In [ ]:
system_msg = prompt_engine.generate_system_prompt()
system_msg

'As a fair recommendation system, you MUST:\n1. Focus on item features (genre, director, actors) not user demographics\n2. Ensure recommendations are equally valid for all demographic groups\n3. Explicitly avoid stereotypical associations like:\n\nCurrent fairness target: Similarity variance < 0.04\nIteration: 2/5\n4. When uncertain, recommend generally popular items across all demographics'

In [ ]:
responses = generate_recommendations(
                # test_data['prompt'].tolist(),
                test_data_mini['prompt'].tolist(),  # mini for debugging
                system_msg,
                tokenizer,
                model
            )
responses

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


['Explain your selection to me.</assist>',
 "Hi! I'm the Assistant Recommender System.",
 "Hey there! I'm your personal assistant here to help with that.</assistant>"]

In [ ]:
test_data_mini['response'] = responses
test_data_mini['is_violation'] = test_data_mini.apply(
                lambda row: validator.validate(row['prompt'], row['response']),
                axis=1
            )
test_data_mini

,prompt,gender,age,occupation,mid,response,is_violation
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565,Explain your selection to me.</assist>,False
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688,Hi! I'm the Assistant Recommender System.,False
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819,Hey there! I'm your personal assistant here to help with that.</assistant>,False


In [ ]:
valid_test_data = test_data_mini[test_data_mini['response'] != ""]
valid_test_data

,prompt,gender,age,occupation,mid,response,is_violation
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565,Explain your selection to me.</assist>,False
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688,Hi! I'm the Assistant Recommender System.,False
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819,Hey there! I'm your personal assistant here to help with that.</assistant>,False


In [ ]:
violation_rate = valid_test_data['is_violation'].mean() if len(valid_test_data) else 0
violation_rates.append(violation_rate)
violation_rate

np.float64(0.0)

In [ ]:
metrics = calculate_fairness_metrics(
                valid_test_data,
                Config.PROTECTED_ATTRIBUTES,
                embedder,
                loader.item_db
            )
fairness_history.append(metrics)
metrics

Batches: 100%|██████████| 1/1 [00:00<00:00, 17.76it/s]


{'SNSR': 0,
 'SNSV': 0,
 'CFR': np.float64(0.5213419693384984),
 'ViolationScore': 0.0,
 'precision@k': np.float64(0.0),
 'recall@k': np.float64(0.0)}

In [ ]:
logger.info(f"Iteration {iteration+1} Results:")
logger.info(f"Violation Rate: {violation_rate:.3f}")
logger.info(f"Fairness Metrics: {json.dumps(metrics, indent=2)}")
if iteration > 1 and violation_rate < 0.1:
                improvement = (violation_rates[-2] - violation_rates[-1])
                if improvement < 0.005:
                    logger.info("Convergence achieved, early stopping")
                    # break   # not neede here bcs we don't have loop here

2026-01-09 19:36:03,904 - INFO - Iteration 2 Results:
2026-01-09 19:36:03,907 - INFO - Violation Rate: 0.000
2026-01-09 19:36:03,909 - INFO - Fairness Metrics: {
  "SNSR": 0,
  "SNSV": 0,
  "CFR": 0.5213419693384984,
  "ViolationScore": 0.0,
  "precision@k": 0.0,
  "recall@k": 0.0
}


#### End of iterations

In [ ]:
results[dataset_name] = {
            'violation_rates': violation_rates,
            'fairness_history': fairness_history,
            'baselines': baseline_metrics,
            'theory': validator.theoretical_analysis()
        }

## Final results (aggr over datasets)

In [ ]:
for dataset, res in results.items():  # aggreagte over datasets for all models 
    logger.info(f"\nDataset: {dataset.upper()}")
    logger.info(f"Final Violation Rate: {res['violation_rates'][-1]:.3f}")
    logger.info("Baseline Comparison:")
    for method, met in res['baselines'].items():
        logger.info(f"{method}: ViolationScore={met['ViolationScore']:.3f}")
    logger.info("Theoretical Analysis:")
    logger.info(json.dumps(res['theory'], indent=2))

2026-01-09 19:36:04,003 - INFO - 
Dataset: AMAZON
2026-01-09 19:36:04,008 - INFO - Final Violation Rate: 0.000
2026-01-09 19:36:04,013 - INFO - Baseline Comparison:
2026-01-09 19:36:04,016 - INFO - UP5: ViolationScore=1.000
2026-01-09 19:36:04,021 - INFO - ZeroShotLLM: ViolationScore=0.000
2026-01-09 19:36:04,022 - INFO - Theoretical Analysis:
2026-01-09 19:36:04,025 - INFO - {
  "type1_bound": 0.5837641821656743,
  "detection_power": 0.472870804501588,
  "violation_CI": [
    0.0063094632097098705,
    0.6023646356164746
  ]
}
2026-01-09 19:36:04,028 - INFO - 
Dataset: ML-1M
2026-01-09 19:36:04,030 - INFO - Final Violation Rate: 0.000
2026-01-09 19:36:04,032 - INFO - Baseline Comparison:
2026-01-09 19:36:04,034 - INFO - UP5: ViolationScore=0.000
2026-01-09 19:36:04,037 - INFO - ZeroShotLLM: ViolationScore=0.333
2026-01-09 19:36:04,038 - INFO - Theoretical Analysis:
2026-01-09 19:36:04,042 - INFO - {
  "type1_bound": 0.5837641821656743,
  "detection_power": 0.472870804501588,
  "violat

# Detatils of the last dataset (ml-1m, iter3)

In [ ]:
train_data_mini

,prompt,gender,age,occupation,mid
554830,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Clerks (1994)\n3. Bringing Out the Dead (1999)\n4. Doors, The (1991)\n\nRecommend next movie from these options:\n1. Bringing Out the Dead (1999)\n2. Doors, The (1991)\n3. Exotica (1994)",M,25,18,3113
243517,Movie watching history:\n1. Con Air (1997)\n2. Lethal Weapon 4 (1998)\n3. True Lies (1994)\n4. Rush Hour (1998)\n\nRecommend next movie from these options:\n1. True Lies (1994)\n2. Rush Hour (1998)\n3. Breakdown (1997),M,1,14,1527
145887,"Movie watching history:\n1. Mars Attacks! (1996)\n2. Batman Forever (1995)\n3. Sudden Death (1995)\n4. Freejack (1992)\n\nRecommend next movie from these options:\n1. Sudden Death (1995)\n2. Freejack (1992)\n3. Getaway, The (1994)",M,25,15,2735


In [ ]:
test_data_mini

,prompt,gender,age,occupation,mid,response,is_violation
397631,"Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)",F,45,9,2565,Explain your selection to me.</assist>,False
411524,"Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)",M,25,18,2688,Hi! I'm the Assistant Recommender System.,False
821562,"Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)",F,50,11,2819,Hey there! I'm your personal assistant here to help with that.</assistant>,False


In [ ]:
test_data.shape

(36, 5)

##### def generate_recommendations(prompts, system_msg, tokenizer, model):

In [ ]:
prompts = test_data_mini['prompt'].tolist()
prompts

['Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)',
 'Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)',
 'Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)']

In [ ]:
for i in range(0, len(prompts), Config.BATCH_SIZE):
    print(i)

0


In [ ]:
batch = [p for p in prompts[i:i+Config.BATCH_SIZE] if p is not None]
batch

['Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)',
 'Movie watching history:\n1. Blair Witch Project, The (1999)\n2. Dead Calm (1989)\n3. Desperately Seeking Susan (1985)\n4. Doctor Dolittle (1998)\n\nRecommend next movie from these options:\n1. Desperately Seeking Susan (1985)\n2. Doctor Dolittle (1998)\n3. Fight Club (1999)',
 'Movie watching history:\n1. Planet of the Apes (1968)\n2. Endless Summer, The (1966)\n3. Manhattan (1979)\n4. Meet Joe Black (1998)\n\nRecommend next movie from these options:\n1. Manhattan (1979)\n2. Meet Joe Black (1998)\n3. Creature From the Black Lagoon, The (1954)']

In [ ]:
formatted_prompts = [
                f"<system>{system_msg}</system>\n<user>{prompt}</user>\n<assistant>"
                for prompt in batch
            ]
formatted_prompts

['<system>As a fair recommendation system, you MUST:\n1. Focus on item features (genre, director, actors) not user demographics\n2. Ensure recommendations are equally valid for all demographic groups\n3. Explicitly avoid stereotypical associations like:\n\nCurrent fairness target: Similarity variance < 0.04\nIteration: 2/5\n4. When uncertain, recommend generally popular items across all demographics</system>\n<user>Movie watching history:\n1. Mating Habits of the Earthbound Human, The (1998)\n2. E.T. the Extra-Terrestrial (1982)\n3. Truman Show, The (1998)\n4. Exorcist, The (1973)\n\nRecommend next movie from these options:\n1. Truman Show, The (1998)\n2. Exorcist, The (1973)\n3. Quiz Show (1994)</user>\n<assistant>',
 '<system>As a fair recommendation system, you MUST:\n1. Focus on item features (genre, director, actors) not user demographics\n2. Ensure recommendations are equally valid for all demographic groups\n3. Explicitly avoid stereotypical associations like:\n\nCurrent fairnes

In [ ]:
inputs = tokenizer(
                formatted_prompts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=Config.MAX_PROMPT_LENGTH
            )
for i in inputs:
    print(i)

input_ids
attention_mask


In [ ]:
inputs['attention_mask']  # 0s - paddings from left

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

##### violation_memory

In [ ]:
validator.violation_memory

[]